# PHD quickstart — spawn one agent

Uses the test stack in `tests/fixtures/stack/` (2 skills, an `echo` MCP server, a `pre_tool` guard hook) so it works out of the box.

* Cells marked **💸** launch a real CLI and cost a few cents (claude/haiku) — or nothing (opencode/space-bunny-free).
* Jupyter already runs an event loop, so use the **async** API here: `await dispatcher.start(...)` / `await handle.wait()`. The blocking `spawn()` / `resume()` are for plain scripts.
* After updating PHD (`git pull`), **restart the kernel** — Jupyter caches imported modules, so a running kernel keeps the old code.
* Kernel: pick the project's `.venv` (install with `uv pip install -e ".[dev,examples]"`).

In [ ]:
from pathlib import Path
import tempfile

import phd
from phd import Credentials, Dispatcher, Inputs, RunStatus, Spec, Stack

REPO = Path.cwd().parent if Path.cwd().name == "examples" else Path.cwd()
FIXTURES = REPO / "tests" / "fixtures"
print("phd", phd.__version__, "| repo:", REPO)

## 1. Pick a harness
`claude` (haiku), `codex`, or `opencode` (free space-bunny-free model). Uses your existing CLI login.

In [ ]:
HARNESS = "claude"
MODELS = {"claude": "haiku", "codex": "gpt-5.6-terra", "opencode": "opencode/space-bunny-free"}

# Restricted autonomy blocks shell for claude/opencode; allow just `echo` for this demo.
ALLOW_ECHO = {
    "claude": {"permissions": {"allow": ["Bash(echo:*)", "PowerShell(echo:*)"]}},
    "codex": {},
    "opencode": {"permission": {"bash": {"echo *": "allow"}}},
}

## 2. The agent (stack) and where it works (spec)

In [ ]:
stack = Stack.load(FIXTURES / "stack")
print(stack.ref, "| agent:", stack.agent_name)
print("skills:", [s.name for s in stack.skills])
print("mcp:", list(stack.mcp_servers), "| hooks:", [(h.event, h.tool) for h in stack.hooks])

workdir = Path(tempfile.mkdtemp(prefix="phd-demo-"))   # the agent's cwd

spec = Spec(
    harness=HARNESS,
    model=MODELS[HARNESS],
    stack=stack,
    cwd=workdir,
    credentials=Credentials.from_host(HARNESS),
    secrets={"ECHO_PREFIX": "phd-echo"},      # fills ${env:ECHO_PREFIX} in the stack's .mcp.json
    harness_config=ALLOW_ECHO[HARNESS],
    timeout_s=300,
)
# Codex refuses a CODEX_HOME under %TEMP%; the default runs root (~/.phd/runs) is fine.
dispatcher = Dispatcher(spec)
print("warnings:", dispatcher.warnings)

## 3. The task

In [ ]:
inputs = Inputs(
    brief=FIXTURES / "inputs" / "brief.md",   # list skills, call the echo MCP, run `echo hello-from-shell`
    rules=FIXTURES / "inputs" / "RULES.md",
)
print((FIXTURES / "inputs" / "brief.md").read_text())

## 4. Dry run — build the isolated config home, launch nothing

In [ ]:
plan = dispatcher.plan(inputs)
print("argv:", plan.argv)
print("env:", plan.redacted_env())
print()
print("files in the run home:")
for p in sorted(plan.home.rglob("*")):
    if p.is_file() and p.relative_to(plan.home).parts[0] != "stack":
        print("  ", p.relative_to(plan.home))
print()
print("first user message (stdin):")
print(plan.stdin)


## 5. 💸 Run it, streaming events

In [ ]:
handle = await dispatcher.start(inputs)
async for ev in handle.events():
    if ev.type in ("message", "tool_call", "usage", "run.finished"):
        print(f"[{ev.type}]", str(ev.data)[:160])
result = await handle.wait()

In [ ]:
print("status :", result.status)
print("cost   :", result.cost_usd, f"({result.cost_source})")
print("tokens :", result.usage)
print("session:", result.session_id)
print("run dir:", result.run_dir)
if result.error:
    print("error  :", result.error.kind, "-", result.error.message)
print()
print(result.result_text or "")
print()
print("hook fired:", (Path(result.run_dir) / "hook.log").is_file())


## 6. 💸 Continue the same conversation

In [ ]:
followup = await (await dispatcher.start_resume(result, "In one sentence: what did the echo tool return?")).wait()
print(followup.status, "| same session:", followup.session_id == result.session_id)
print(followup.result_text)

## 7. The run ledger

In [ ]:
for r in phd.runs.list()[-5:]:
    print(r.run_id, r.harness, r.status, r.cost_usd)

## 8. Try your own stack

```python
stack = Stack.load("path/to/your/config")   # stack.json, agents/<one>.md, skills/, .mcp.json, hooks/hooks.json
spec = Spec(harness="claude", model="sonnet", stack=stack, cwd="path/to/repo",
            credentials=Credentials.from_host("claude"))
```